In [0]:
import requests

In [0]:
import requests

class JiraClient:
   
    def __init__(self, base_url, version, auth, headers):
        self.base_url = base_url
        self.version = version
        self.auth = auth
        self.headers = headers

    def build_url(self, endpoint):
        if not self.base_url or not self.version or not endpoint:
            raise ValueError("Invalid url components")

        return f"{self.base_url}{self.version}{endpoint}"
    
    def fetch(self, endpoint, params, data_key, batch_size=100):
        
        url = self.build_url(endpoint)

        all_data = []
        start = 0

        while True:
            params_local = params.copy() if params else {}
            params_local["startAt"] = start
            params_local["maxResults"] = batch_size

            response = requests.get(
                url,
                auth=self.auth,
                headers=self.headers,
                params=params_local
            )
            response.raise_for_status()

            data = response.json()
            items = data.get(data_key, [])

            all_data.extend(items)

            if len(items) < batch_size:
                break

            start += batch_size

        return all_data

    def get_projects(self):
        return self.fetch(
            endpoint="/project/search",
            params={},
            data_key="values"
        )

    def get_issues(self, jql, fields):
        return self.fetch(
            endpoint="/search/jql",
            params={
                "jql": jql,
                "fields": fields
            },
            data_key="issues"
        )
